In [22]:
# ============================================================
# M1: GLOBAL + PERSONALIZED PROTOTYPE METRIC LEARNER
#
# FINAL HYPERNETWORK-FREE MODEL
#
# Motivation:
#
# H7/H8/H10 showed that:
#
#   - label-aware support information helps
#   - minority-aware training helps
#   - HD remains extremely difficult
#   - HyperNetwork-generated classifier is not exploiting
#     prototype information sufficiently
#
# M1 removes the HyperNetwork entirely.
#
# Architecture:
#
#                  1024-D clip
#                       |
#                       v
#                Shared Encoder
#                       |
#                       v
#                  128-D embedding
#                       |
#          +------------+-------------+
#          |                          |
#     Global prototypes       Student support
#          |                          |
#          +------------+-------------+
#                       |
#                       v
#              Personalized prototypes
#                       |
#                       v
#             Distance-based classifier
#
#
# IMPORTANT TEST PROTOCOL:
#
#     unseen student
#          |
#     real first K clips
#       OR real random K clips
#          |
#     personalization
#          |
#     remaining real clips
#
# NO test balancing.
# NO synthetic examples.
# NO fabricated HD support.
#
#
# M1 COMPONENTS
#
# 1. Shared embedding encoder
# 2. Global class prototypes
# 3. Student-specific prototype adaptation
# 4. Supervised contrastive loss
# 5. Explicit HD-vs-D margin loss
# 6. Ordinal loss
# 7. Mild class-balanced CE
#
# ============================================================

import argparse
import json
import os
import re
import random
from collections import Counter

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    f1_score,
    precision_recall_fscore_support,
    confusion_matrix,
)
from sklearn.preprocessing import StandardScaler


# ============================================================
# CONFIGURATION
# ============================================================

LABELS = [
    "Highly Disengage",
    "Disengage",
    "Engage",
    "Highly Engage",
]

LABEL_TO_ID = {
    label: i
    for i, label in enumerate(LABELS)
}

NUM_CLASSES = 4
INPUT_DIM = 1024

FEATURE_COLS = [
    f"emb_{i}"
    for i in range(INPUT_DIM)
]


# ============================================================
# MODEL DIMENSIONS
# ============================================================

EMBED_DIM = 128

ENCODER_HIDDEN = 256

PROTOTYPE_DIM = EMBED_DIM


# ============================================================
# TRAINING
# ============================================================

DEFAULT_EPISODES = 2000

DEFAULT_SEED = 42

LEARNING_RATE = 7e-4

WEIGHT_DECAY = 1e-4

GRAD_CLIP = 5.0


# ============================================================
# EPISODIC TRAINING
# ============================================================

TRAIN_K_VALUES = [
    5,
    10,
    20,
]

QUERY_MAX = 64


# ============================================================
# LOSS WEIGHTS
#
# Keep these moderate.
#
# We do NOT want H10-style aggressive reweighting.
# ============================================================

CE_WEIGHT = 1.0

CONTRASTIVE_WEIGHT = 0.12

HD_MARGIN_WEIGHT = 0.40

ORDINAL_WEIGHT = 0.08


# ============================================================
# CONTRASTIVE
# ============================================================

CONTRASTIVE_TEMPERATURE = 0.10


# ============================================================
# HD VS D MARGIN
# ============================================================

HD_MARGIN = 0.20


# ============================================================
# PROTOTYPE ADAPTATION
# ============================================================

# When a student has a class-specific support example:
#
# personalized =
#     alpha * student prototype
#     +
#     (1-alpha) * global prototype
#
# This prevents a tiny support sample from completely
# destroying the globally learned representation.

SUPPORT_ALPHA = 0.65


# ============================================================
# REPRODUCIBILITY
# ============================================================

def set_seed(seed):

    random.seed(seed)

    np.random.seed(seed)

    torch.manual_seed(seed)

    if torch.cuda.is_available():

        torch.cuda.manual_seed_all(seed)


# ============================================================
# DATA LOADING
# ============================================================

def parse_group_from_clip_id(clip_id):

    """
    Expected:

        videoX_clipindex_person...

    Example:

        video123_17_person4

    becomes:

        video123_person4
    """

    match = re.match(
        r"^(video[^_]+)_(\d+)_((?:person).+)$",
        str(clip_id),
    )

    if not match:

        return None

    return (
        f"{match.group(1)}_"
        f"{match.group(3)}"
    )


def load_clip_dataframe(json_path):

    with open(
        json_path,
        "r",
        encoding="utf-8",
    ) as f:

        data = json.load(f)

    records = []

    for clip_id, meta in data.items():

        if not isinstance(meta, dict):

            continue

        label = meta.get("label")

        if label not in LABELS:

            continue

        embeds = meta.get("embeds")

        if not isinstance(
            embeds,
            list,
        ):

            continue

        if len(embeds) != INPUT_DIM:

            continue

        group = parse_group_from_clip_id(
            clip_id
        )

        if group is None:

            continue

        arr = np.asarray(
            embeds,
            dtype=np.float32,
        )

        if not np.all(
            np.isfinite(arr)
        ):

            continue

        clip_index_match = re.search(
            r"_(\d+)_",
            str(clip_id),
        )

        clip_index = (
            int(
                clip_index_match.group(1)
            )
            if clip_index_match
            else 0
        )

        records.append(
            {
                "clip_id": str(clip_id),

                "group": group,

                "label": str(label),

                "label_id":
                    LABEL_TO_ID[str(label)],

                "clip_index":
                    clip_index,

                "embedding": arr,
            }
        )

    if not records:

        raise ValueError(
            f"No valid labelled clips "
            f"found in {json_path}"
        )

    df = pd.DataFrame(
        records
    )

    emb_matrix = np.vstack(
        df["embedding"].to_numpy()
    )

    emb_df = pd.DataFrame(
        emb_matrix,
        columns=FEATURE_COLS,
        index=df.index,
    )

    df = pd.concat(
        [
            df.drop(
                columns=["embedding"]
            ),
            emb_df,
        ],
        axis=1,
    )

    return df


# ============================================================
# FOLD LOADING
# ============================================================

def load_group_fold_map(
    fold_csv_path
):

    fold_df = pd.read_csv(
        fold_csv_path
    )

    required = {
        "group",
        "fold",
    }

    missing = (
        required
        - set(fold_df.columns)
    )

    if missing:

        raise ValueError(
            "Fold CSV missing required "
            f"columns: {sorted(missing)}"
        )

    group_fold = (
        fold_df[
            ["group", "fold"]
        ]
        .drop_duplicates()
    )

    check = (
        group_fold
        .groupby("group")["fold"]
        .nunique()
    )

    bad = check[
        check != 1
    ]

    if len(bad) > 0:

        raise ValueError(
            "Groups assigned to multiple "
            f"folds: {bad.index.tolist()}"
        )

    return dict(
        zip(
            group_fold["group"],
            group_fold["fold"].astype(int),
        )
    )


# ============================================================
# REAL TEST SUPPORT SAMPLING
# ============================================================

def sample_support_query(
    student_df,
    k_shot,
    protocol,
    rng,
):

    """
    NEVER change this function to balance classes.

    This is the real evaluation protocol.
    """

    ordered = (
        student_df
        .sort_values("clip_index")
        .copy()
    )

    if len(ordered) <= k_shot:

        return (
            None,
            None,
            f"insufficient_clips_n="
            f"{len(ordered)}_k={k_shot}",
        )

    if protocol == "early":

        support = ordered.iloc[
            :k_shot
        ].copy()

    elif protocol == "random":

        sampled_idx = rng.choice(
            ordered.index.to_numpy(),
            size=k_shot,
            replace=False,
        )

        support = ordered.loc[
            sampled_idx
        ].copy()

    else:

        return (
            None,
            None,
            f"unknown_protocol_{protocol}",
        )

    query = ordered.loc[
        ~ordered.index.isin(
            support.index
        )
    ].copy()

    if query.empty:

        return (
            None,
            None,
            "empty_query",
        )

    return (
        support,
        query,
        None,
    )


# ============================================================
# TRAINING SUPPORT SAMPLER
# ============================================================

def sample_balanced_support(
    student_df,
    k_shot,
    rng,
):

    """
    TRAINING ONLY.

    No synthetic examples.

    Try to expose the encoder to all available classes.
    """

    class_indices = {}

    available_classes = []

    for class_id in range(
        NUM_CLASSES
    ):

        idx = (
            student_df[
                student_df["label_id"]
                == class_id
            ]
            .index
            .to_numpy()
        )

        if len(idx) > 0:

            class_indices[
                class_id
            ] = idx

            available_classes.append(
                class_id
            )

    if not available_classes:

        return None

    n_classes = len(
        available_classes
    )

    allocation = {
        c: k_shot // n_classes
        for c in available_classes
    }

    remainder = (
        k_shot % n_classes
    )

    classes_for_remainder = (
        available_classes.copy()
    )

    rng.shuffle(
        classes_for_remainder
    )

    for c in classes_for_remainder[
        :remainder
    ]:

        allocation[c] += 1

    selected = []

    unused = 0

    for c in available_classes:

        requested = allocation[c]

        available = class_indices[c]

        take = min(
            requested,
            len(available),
        )

        unused += (
            requested - take
        )

        if take > 0:

            chosen = rng.choice(
                available,
                size=take,
                replace=False,
            )

            selected.extend(
                chosen.tolist()
            )

    # --------------------------------------------------------
    # Fill remaining slots
    # --------------------------------------------------------

    selected_set = set(
        selected
    )

    remaining = []

    for c in available_classes:

        for idx in class_indices[c]:

            if idx not in selected_set:

                remaining.append(idx)

    if unused > 0 and remaining:

        take = min(
            unused,
            len(remaining),
        )

        extra = rng.choice(
            np.asarray(remaining),
            size=take,
            replace=False,
        )

        selected.extend(
            extra.tolist()
        )

    if not selected:

        return None

    return student_df.loc[
        selected
    ].copy()


# ============================================================
# TRAINING QUERY SAMPLER
# ============================================================

def sample_balanced_query(
    query_df,
    max_samples,
    rng,
):

    """
    TRAINING ONLY.

    Approximately balance classes.

    Test queries are NEVER modified.
    """

    if query_df.empty:

        return query_df

    class_indices = {}

    available_classes = []

    for class_id in range(
        NUM_CLASSES
    ):

        idx = (
            query_df[
                query_df["label_id"]
                == class_id
            ]
            .index
            .to_numpy()
        )

        if len(idx) > 0:

            class_indices[
                class_id
            ] = idx

            available_classes.append(
                class_id
            )

    if not available_classes:

        return query_df.iloc[0:0]

    n_classes = len(
        available_classes
    )

    per_class = max_samples // n_classes

    selected = []

    for c in available_classes:

        idx = class_indices[c]

        take = min(
            per_class,
            len(idx),
        )

        if take > 0:

            chosen = rng.choice(
                idx,
                size=take,
                replace=False,
            )

            selected.extend(
                chosen.tolist()
            )

    # --------------------------------------------------------
    # Fill leftover slots
    # --------------------------------------------------------

    selected_set = set(
        selected
    )

    remaining = []

    for c in available_classes:

        for idx in class_indices[c]:

            if idx not in selected_set:

                remaining.append(idx)

    remaining_slots = (
        max_samples
        - len(selected)
    )

    if (
        remaining_slots > 0
        and remaining
    ):

        take = min(
            remaining_slots,
            len(remaining),
        )

        extra = rng.choice(
            np.asarray(remaining),
            size=take,
            replace=False,
        )

        selected.extend(
            extra.tolist()
        )

    return query_df.loc[
        selected
    ].copy()


# ============================================================
# SHARED EMBEDDING ENCODER
# ============================================================

class M1Encoder(nn.Module):

    def __init__(
        self,
        input_dim=INPUT_DIM,
        hidden_dim=ENCODER_HIDDEN,
        embed_dim=EMBED_DIM,
    ):

        super().__init__()

        self.net = nn.Sequential(

            nn.Linear(
                input_dim,
                hidden_dim,
            ),

            nn.GELU(),

            nn.LayerNorm(
                hidden_dim
            ),

            nn.Dropout(
                0.10
            ),

            nn.Linear(
                hidden_dim,
                embed_dim,
            ),

            nn.GELU(),

            nn.LayerNorm(
                embed_dim
            ),
        )

    def forward(self, x):

        z = self.net(x)

        # ----------------------------------------------------
        # Normalize for metric learning.
        # ----------------------------------------------------

        z = F.normalize(
            z,
            p=2,
            dim=-1,
        )

        return z


# ============================================================
# GLOBAL PROTOTYPE MEMORY
# ============================================================

class GlobalPrototypeBank(
    nn.Module
):

    """
    Four learnable global anchors.

        HD
        D
        E
        HE

    These are particularly important when a test student's
    support set contains no HD examples.
    """

    def __init__(
        self,
        num_classes=NUM_CLASSES,
        dim=PROTOTYPE_DIM,
    ):

        super().__init__()

        self.prototypes = nn.Parameter(
            torch.randn(
                num_classes,
                dim,
            )
        )

        with torch.no_grad():

            self.prototypes.copy_(
                F.normalize(
                    self.prototypes,
                    dim=1,
                )
            )

    def forward(self):

        return F.normalize(
            self.prototypes,
            dim=1,
        )


# ============================================================
# GLOBAL PROTOTYPE INITIALIZATION
# ============================================================

def initialize_global_prototypes(
    model,
    train_df,
    scaler,
    device,
):

    """
    Initialize global prototypes from the actual training
    embeddings.

    This is only training data.

    No test information is used.
    """

    model.eval()

    with torch.no_grad():

        for class_id in range(
            NUM_CLASSES
        ):

            class_df = train_df[
                train_df["label_id"]
                == class_id
            ]

            if class_df.empty:

                continue

            x_np = scaler.transform(
                class_df[
                    FEATURE_COLS
                ].to_numpy(
                    dtype=np.float32
                )
            )

            x = torch.tensor(
                x_np,
                dtype=torch.float32,
                device=device,
            )

            z = model.encoder(
                x
            )

            prototype = F.normalize(
                z.mean(
                    dim=0
                ),
                dim=0,
            )

            with torch.no_grad():

                model.prototype_bank.prototypes[
                    class_id
                ].copy_(
                    prototype
                )


# ============================================================
# PERSONALIZED PROTOTYPES
# ============================================================

def build_personalized_prototypes(
    global_prototypes,
    support_embeddings,
    support_labels,
    alpha=SUPPORT_ALPHA,
):

    """
    Build student-specific prototypes.

    If a class exists in support:

        personalized =
            alpha * support prototype
            +
            (1-alpha) * global prototype

    If absent:

        personalized =
            global prototype

    Thus:

        HD absent from support
             ↓
        global HD remains available

    This is essential for your dataset.
    """

    prototypes = global_prototypes.clone()

    support_counts = torch.zeros(
        NUM_CLASSES,
        device=support_embeddings.device,
    )

    for class_id in range(
        NUM_CLASSES
    ):

        mask = (
            support_labels
            == class_id
        )

        if mask.any():

            local = F.normalize(
                support_embeddings[
                    mask
                ].mean(
                    dim=0
                ),
                dim=0,
            )

            combined = (
                alpha * local
                +
                (1.0 - alpha)
                * global_prototypes[
                    class_id
                ]
            )

            prototypes[
                class_id
            ] = F.normalize(
                combined,
                dim=0,
            )

            support_counts[
                class_id
            ] = mask.sum()

    return (
        F.normalize(
            prototypes,
            dim=1,
        ),
        support_counts,
    )


# ============================================================
# DISTANCE LOGITS
# ============================================================

def prototype_logits(
    query_embeddings,
    prototypes,
    temperature=0.07,
):

    """
    Cosine similarity converted to logits.

    Higher similarity = higher class score.
    """

    similarity = (
        query_embeddings
        @ prototypes.T
    )

    return (
        similarity
        / temperature
    )


# ============================================================
# SUPERVISED CONTRASTIVE LOSS
# ============================================================

def supervised_contrastive_loss(
    embeddings,
    labels,
    temperature=CONTRASTIVE_TEMPERATURE,
):

    """
    Robust implementation.

    embeddings:
        [N, D]

    labels:
        [N]

    Positive examples:
        same class

    Self-pairs removed.

    If there are no positive pairs, return zero.

    """

    if embeddings.ndim != 2:

        raise ValueError(
            "Embeddings must be [N,D]"
        )

    n = embeddings.shape[0]

    if labels.shape[0] != n:

        raise ValueError(
            "Contrastive loss label/"
            "embedding mismatch: "
            f"embeddings={n}, "
            f"labels={labels.shape[0]}"
        )

    if n < 2:

        return embeddings.sum() * 0.0

    z = F.normalize(
        embeddings,
        dim=1,
    )

    logits = (
        z @ z.T
    ) / temperature

    # --------------------------------------------------------
    # Remove self similarity
    # --------------------------------------------------------

    self_mask = torch.eye(
        n,
        dtype=torch.bool,
        device=embeddings.device,
    )

    logits = logits.masked_fill(
        self_mask,
        -1e9,
    )

    # --------------------------------------------------------
    # Positive mask
    # --------------------------------------------------------

    positive_mask = (
        labels.unsqueeze(0)
        ==
        labels.unsqueeze(1)
    )

    positive_mask = (
        positive_mask
        & ~self_mask
    )

    # --------------------------------------------------------
    # Log-softmax
    # --------------------------------------------------------

    log_prob = F.log_softmax(
        logits,
        dim=1,
    )

    positive_count = (
        positive_mask.sum(
            dim=1
        )
    )

    valid = (
        positive_count > 0
    )

    if not valid.any():

        return embeddings.sum() * 0.0

    mean_log_prob_pos = (
        (
            log_prob
            * positive_mask.float()
        )
        .sum(dim=1)
        /
        positive_count.clamp(
            min=1
        )
    )

    return -mean_log_prob_pos[
        valid
    ].mean()


# ============================================================
# HD VS D MARGIN LOSS
# ============================================================

def hd_d_margin_loss(
    query_embeddings,
    query_labels,
    prototypes,
    margin=HD_MARGIN,
):

    """
    Explicitly separates:

        Highly Disengage
              vs
        Disengage

    This is the most targeted part of M1.

    For HD:

        sim(query, HD)
            >=
        sim(query, D) + margin

    For D:

        sim(query, D)
            >=
        sim(query, HD) + margin
    """

    if query_embeddings.shape[0] == 0:

        return query_embeddings.sum() * 0.0

    hd_mask = (
        query_labels == 0
    )

    d_mask = (
        query_labels == 1
    )

    losses = []

    if hd_mask.any():

        hd_q = query_embeddings[
            hd_mask
        ]

        hd_sim = (
            hd_q
            @ prototypes[0]
        )

        d_sim = (
            hd_q
            @ prototypes[1]
        )

        losses.append(
            F.relu(
                margin
                - hd_sim
                + d_sim
            ).mean()
        )

    if d_mask.any():

        d_q = query_embeddings[
            d_mask
        ]

        d_sim = (
            d_q
            @ prototypes[1]
        )

        hd_sim = (
            d_q
            @ prototypes[0]
        )

        losses.append(
            F.relu(
                margin
                - d_sim
                + hd_sim
            ).mean()
        )

    if not losses:

        return query_embeddings.sum() * 0.0

    return torch.stack(
        losses
    ).mean()


# ============================================================
# ORDINAL REGULARIZATION
# ============================================================

def ordinal_loss(
    logits,
    labels,
):

    """
    Engagement classes are ordinal:

        0 HD
        1 D
        2 E
        3 HE

    Convert 4-class prediction into three cumulative
    binary questions:

        P(y > 0)
        P(y > 1)
        P(y > 2)

    This encourages nearby ordinal states to remain
    structurally meaningful without forcing hard hierarchy.
    """

    if logits.shape[0] == 0:

        return logits.sum() * 0.0

    cumulative_targets = []

    cumulative_logits = []

    for threshold in range(
        NUM_CLASSES - 1
    ):

        target = (
            labels > threshold
        ).float()

        # probability of classes above threshold
        threshold_logit = (
            torch.logsumexp(
                logits[
                    :,
                    threshold + 1:
                ],
                dim=1,
            )
            -
            logits[
                :,
                :threshold + 1
            ].mean(
                dim=1
            )
        )

        cumulative_targets.append(
            target
        )

        cumulative_logits.append(
            threshold_logit
        )

    total = 0.0

    for logit, target in zip(
        cumulative_logits,
        cumulative_targets,
    ):

        total = total + F.binary_cross_entropy_with_logits(
            logit,
            target,
        )

    return (
        total
        /
        (NUM_CLASSES - 1)
    )


# ============================================================
# CLASS-BALANCED CE WEIGHTS
# ============================================================

def compute_training_class_weights(
    train_df,
    device,
):

    """
    Mild inverse-frequency weighting.

    IMPORTANT:
        weights are deliberately capped.

    This is not aggressive H10-style weighting.
    """

    counts = np.zeros(
        NUM_CLASSES,
        dtype=np.float64,
    )

    for class_id in range(
        NUM_CLASSES
    ):

        counts[class_id] = (
            (
                train_df["label_id"]
                == class_id
            )
            .sum()
        )

    safe_counts = np.maximum(
        counts,
        1.0,
    )

    weights = (
        np.sqrt(
            safe_counts.sum()
            /
            (
                NUM_CLASSES
                * safe_counts
            )
        )
    )

    # --------------------------------------------------------
    # Cap maximum weight.
    # --------------------------------------------------------

    weights = np.clip(
        weights,
        0.50,
        2.50,
    )

    weights = (
        weights
        /
        weights.mean()
    )

    return torch.tensor(
        weights,
        dtype=torch.float32,
        device=device,
    )


# ============================================================
# FULL M1 MODEL
# ============================================================

class M1ProtoMetricModel(
    nn.Module
):

    def __init__(self):

        super().__init__()

        self.encoder = M1Encoder()

        self.prototype_bank = (
            GlobalPrototypeBank()
        )

    def encode(self, x):

        return self.encoder(x)

    def global_prototypes(self):

        return self.prototype_bank()


# ============================================================
# EPISODIC TRAINING
# ============================================================

def episodic_train(
    model,
    train_df,
    scaler,
    episodes,
    seed,
    device,
):

    model.train()

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    scheduler = (
        torch.optim.lr_scheduler.CosineAnnealingLR(
            optimizer,
            T_max=max(
                episodes,
                1,
            ),
        )
    )

    rng = np.random.default_rng(
        seed
    )

    groups = sorted(
        train_df["group"].unique()
    )

    if not groups:

        raise ValueError(
            "No training students."
        )

    class_weights = (
        compute_training_class_weights(
            train_df,
            device,
        )
    )

    successful = 0

    running_losses = []

    for episode in range(
        episodes
    ):

        # ----------------------------------------------------
        # Sample student
        # ----------------------------------------------------

        group = rng.choice(
            groups
        )

        student_df = train_df[
            train_df["group"]
            == group
        ].copy()

        k = int(
            rng.choice(
                TRAIN_K_VALUES
            )
        )

        if len(student_df) <= k:

            continue

        # ----------------------------------------------------
        # Balanced support
        # ----------------------------------------------------

        support_df = (
            sample_balanced_support(
                student_df,
                k,
                rng,
            )
        )

        if (
            support_df is None
            or support_df.empty
        ):

            continue

        # ----------------------------------------------------
        # Remaining query
        # ----------------------------------------------------

        query_df = student_df.loc[
            ~student_df.index.isin(
                support_df.index
            )
        ].copy()

        if query_df.empty:

            continue

        query_df = (
            sample_balanced_query(
                query_df,
                QUERY_MAX,
                rng,
            )
        )

        if query_df.empty:

            continue

        # ----------------------------------------------------
        # Transform
        # ----------------------------------------------------

        support_x_np = scaler.transform(
            support_df[
                FEATURE_COLS
            ].to_numpy(
                dtype=np.float32
            )
        )

        query_x_np = scaler.transform(
            query_df[
                FEATURE_COLS
            ].to_numpy(
                dtype=np.float32
            )
        )

        support_y_np = (
            support_df[
                "label_id"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        query_y_np = (
            query_df[
                "label_id"
            ]
            .to_numpy(
                dtype=np.int64
            )
        )

        support_x = torch.tensor(
            support_x_np,
            dtype=torch.float32,
            device=device,
        )

        support_y = torch.tensor(
            support_y_np,
            dtype=torch.long,
            device=device,
        )

        query_x = torch.tensor(
            query_x_np,
            dtype=torch.float32,
            device=device,
        )

        query_y = torch.tensor(
            query_y_np,
            dtype=torch.long,
            device=device,
        )

        # ----------------------------------------------------
        # Encode support + query
        # ----------------------------------------------------

        support_z = model.encode(
            support_x
        )

        query_z = model.encode(
            query_x
        )

        # ----------------------------------------------------
        # Global prototypes
        # ----------------------------------------------------

        global_prototypes = (
            model.global_prototypes()
        )

        # ----------------------------------------------------
        # Personalized prototypes
        # ----------------------------------------------------

        prototypes, support_counts = (
            build_personalized_prototypes(
                global_prototypes,
                support_z,
                support_y,
            )
        )

        # ----------------------------------------------------
        # Prototype logits
        # ----------------------------------------------------

        logits = prototype_logits(
            query_z,
            prototypes,
        )

        # ----------------------------------------------------
        # LOSS 1:
        # Mild class-balanced CE
        # ----------------------------------------------------

        loss_ce = F.cross_entropy(
            logits,
            query_y,
            weight=class_weights,
        )

        # ----------------------------------------------------
        # LOSS 2:
        # Supervised contrastive
        #
        # IMPORTANT:
        # support + query labels are both available during
        # META-TRAINING.
        #
        # This does not use test labels.
        # ----------------------------------------------------

        all_z = torch.cat(
            [
                support_z,
                query_z,
            ],
            dim=0,
        )

        all_y = torch.cat(
            [
                support_y,
                query_y,
            ],
            dim=0,
        )

        loss_contrastive = (
            supervised_contrastive_loss(
                all_z,
                all_y,
            )
        )

        # ----------------------------------------------------
        # LOSS 3:
        # Explicit HD vs D
        # ----------------------------------------------------

        loss_hd_d = (
            hd_d_margin_loss(
                query_z,
                query_y,
                prototypes,
            )
        )

        # ----------------------------------------------------
        # LOSS 4:
        # Ordinal structure
        # ----------------------------------------------------

        loss_ordinal = (
            ordinal_loss(
                logits,
                query_y,
            )
        )

        # ----------------------------------------------------
        # TOTAL
        # ----------------------------------------------------

        loss = (
            CE_WEIGHT
            * loss_ce
            +
            CONTRASTIVE_WEIGHT
            * loss_contrastive
            +
            HD_MARGIN_WEIGHT
            * loss_hd_d
            +
            ORDINAL_WEIGHT
            * loss_ordinal
        )

        # ----------------------------------------------------
        # Backprop
        # ----------------------------------------------------

        optimizer.zero_grad(
            set_to_none=True
        )

        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(),
            GRAD_CLIP,
        )

        optimizer.step()

        scheduler.step()

        successful += 1

        running_losses.append(
            float(
                loss.item()
            )
        )

        # ----------------------------------------------------
        # Logging
        # ----------------------------------------------------

        if (
            (episode + 1) % 100
            == 0
        ):

            mean_loss = (
                np.mean(
                    running_losses[-100:]
                )
            )

            print(
                f"    Episode "
                f"{episode + 1}/"
                f"{episodes} "
                f"loss={mean_loss:.4f} "
                f"CE={loss_ce.item():.4f} "
                f"CON={loss_contrastive.item():.4f} "
                f"HD-D={loss_hd_d.item():.4f} "
                f"ORD={loss_ordinal.item():.4f}"
            )

    print(
        f"    Successful episodes: "
        f"{successful}/{episodes}"
    )

    if running_losses:

        print(
            f"    Final mean loss: "
            f"{np.mean(running_losses[-100:]):.4f}"
        )

    return model


# ============================================================
# METRICS
# ============================================================

def calculate_metrics(
    y_true,
    y_pred,
):

    accuracy = accuracy_score(
        y_true,
        y_pred,
    )

    balanced_accuracy = (
        balanced_accuracy_score(
            y_true,
            y_pred,
        )
    )

    macro_f1 = f1_score(
        y_true,
        y_pred,
        labels=list(
            range(NUM_CLASSES)
        ),
        average="macro",
        zero_division=0,
    )

    precision, recall, f1, support = (
        precision_recall_fscore_support(
            y_true,
            y_pred,
            labels=list(
                range(NUM_CLASSES)
            ),
            zero_division=0,
        )
    )

    cm = confusion_matrix(
        y_true,
        y_pred,
        labels=list(
            range(NUM_CLASSES)
        ),
    )

    return {
        "accuracy":
            float(accuracy),

        "balanced_accuracy":
            float(balanced_accuracy),

        "macro_f1":
            float(macro_f1),

        "precision":
            precision,

        "recall":
            recall,

        "f1":
            f1,

        "support":
            support,

        "confusion_matrix":
            cm,
    }


# ============================================================
# MAJORITY BASELINE
# ============================================================

def majority_prediction(
    y_query
):

    counts = Counter(
        y_query.tolist()
    )

    majority_class = max(
        counts,
        key=counts.get,
    )

    return np.full(
        len(y_query),
        majority_class,
        dtype=np.int64,
    )


# ============================================================
# PROTOTYPE BASELINE
# ============================================================

def evaluate_student(
    model,
    support_df,
    query_df,
    scaler,
    device,
):

    model.eval()

    # --------------------------------------------------------
    # Transform
    # --------------------------------------------------------

    support_x_np = scaler.transform(
        support_df[
            FEATURE_COLS
        ].to_numpy(
            dtype=np.float32
        )
    )

    query_x_np = scaler.transform(
        query_df[
            FEATURE_COLS
        ].to_numpy(
            dtype=np.float32
        )
    )

    support_y_np = (
        support_df[
            "label_id"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    y_true = (
        query_df[
            "label_id"
        ]
        .to_numpy(
            dtype=np.int64
        )
    )

    support_x = torch.tensor(
        support_x_np,
        dtype=torch.float32,
        device=device,
    )

    support_y = torch.tensor(
        support_y_np,
        dtype=torch.long,
        device=device,
    )

    query_x = torch.tensor(
        query_x_np,
        dtype=torch.float32,
        device=device,
    )

    # --------------------------------------------------------
    # Encode
    # --------------------------------------------------------

    with torch.no_grad():

        support_z = model.encode(
            support_x
        )

        query_z = model.encode(
            query_x
        )

        global_prototypes = (
            model.global_prototypes()
        )

        personalized_prototypes, counts = (
            build_personalized_prototypes(
                global_prototypes,
                support_z,
                support_y,
            )
        )

        logits = prototype_logits(
            query_z,
            personalized_prototypes,
        )

        y_pred = (
            torch.argmax(
                logits,
                dim=1,
            )
            .cpu()
            .numpy()
        )

        probabilities = F.softmax(
            logits,
            dim=1,
        ).cpu().numpy()

    metrics = calculate_metrics(
        y_true,
        y_pred,
    )

    majority_pred = (
        majority_prediction(
            y_true
        )
    )

    majority_metrics = (
        calculate_metrics(
            y_true,
            majority_pred,
        )
    )

    # --------------------------------------------------------
    # Prototype-only diagnostic
    # --------------------------------------------------------

    available_classes = int(
        (
            counts > 0
        )
        .sum()
        .item()
    )

    return (
        y_true,
        y_pred,
        probabilities,
        metrics,
        majority_metrics,
        available_classes,
        counts.cpu().numpy(),
    )


# ============================================================
# RUN EXPERIMENT
# ============================================================

def run_experiment(
    clip_df,
    group_fold,
    k_values,
    seed,
    episodes,
    device,
    output_dir,
):

    eligible_groups = set(
        group_fold.keys()
    )

    eval_df = clip_df[
        clip_df["group"].isin(
            eligible_groups
        )
    ].copy()

    summary_rows = []

    per_student_rows = []

    per_class_rows = []

    class_summary_rows = []

    confusion_rows = []

    skipped_rows = []

    # --------------------------------------------------------
    # Folds
    # --------------------------------------------------------

    fold_ids = sorted(
        set(
            group_fold.values()
        )
    )

    for fold_id in fold_ids:

        print()
        print("=" * 80)
        print(
            f"M1 OUTER FOLD {fold_id}"
        )
        print("=" * 80)

        test_groups = [
            g
            for g, f in group_fold.items()
            if f == fold_id
        ]

        train_groups = [
            g
            for g, f in group_fold.items()
            if f != fold_id
        ]

        train_df = eval_df[
            eval_df["group"].isin(
                train_groups
            )
        ].copy()

        test_df = eval_df[
            eval_df["group"].isin(
                test_groups
            )
        ].copy()

        # ----------------------------------------------------
        # Leakage check
        # ----------------------------------------------------

        overlap = (
            set(train_df["group"])
            &
            set(test_df["group"])
        )

        if overlap:

            raise RuntimeError(
                f"DATA LEAKAGE in fold "
                f"{fold_id}: "
                f"{overlap}"
            )

        print(
            f"Training students: "
            f"{train_df['group'].nunique()}"
        )

        print(
            f"Test students: "
            f"{test_df['group'].nunique()}"
        )

        print()
        print(
            "Training class distribution:"
        )

        print(
            train_df[
                "label"
            ].value_counts()
        )

        # ----------------------------------------------------
        # Scaler ONLY training data
        # ----------------------------------------------------

        scaler = StandardScaler()

        X_train = (
            train_df[
                FEATURE_COLS
            ]
            .to_numpy(
                dtype=np.float32
            )
        )

        scaler.fit(
            X_train
        )

        # ----------------------------------------------------
        # Fresh model
        # ----------------------------------------------------

        set_seed(
            seed + fold_id
        )

        model = (
            M1ProtoMetricModel()
        )

        model.to(device)

        # ----------------------------------------------------
        # Initialize global prototypes from TRAINING data
        # ----------------------------------------------------

        initialize_global_prototypes(
            model,
            train_df,
            scaler,
            device,
        )

        print()
        print(
            f"M1 meta-training "
            f"({episodes} episodes)"
        )

        episodic_train(
            model=model,
            train_df=train_df,
            scaler=scaler,
            episodes=episodes,
            seed=(
                seed
                + fold_id * 1000
            ),
            device=device,
        )

        # ----------------------------------------------------
        # Test students
        # ----------------------------------------------------

        for group in test_groups:

            student_df = test_df[
                test_df["group"]
                == group
            ].copy()

            if student_df.empty:

                continue

            for k in k_values:

                for protocol in [
                    "random",
                    "early",
                ]:

                    local_seed = (
                        seed
                        + fold_id * 10000
                        + k * 100
                        + sum(
                            ord(ch)
                            for ch in group
                        )
                        + (
                            1
                            if protocol
                            == "early"
                            else 0
                        )
                    )

                    rng = (
                        np.random.default_rng(
                            local_seed
                        )
                    )

                    (
                        support_df,
                        query_df,
                        reason,
                    ) = (
                        sample_support_query(
                            student_df,
                            k,
                            protocol,
                            rng,
                        )
                    )

                    if reason is not None:

                        skipped_rows.append(
                            {
                                "method":
                                    "M1_proto_metric",

                                "outer_fold":
                                    fold_id,

                                "student_group":
                                    group,

                                "k_shot":
                                    k,

                                "support_protocol":
                                    protocol,

                                "reason":
                                    reason,
                            }
                        )

                        continue

                    (
                        y_true,
                        y_pred,
                        probabilities,
                        metrics,
                        majority_metrics,
                        prototype_classes_available,
                        support_counts,
                    ) = evaluate_student(
                        model=model,
                        support_df=support_df,
                        query_df=query_df,
                        scaler=scaler,
                        device=device,
                    )

                    # ------------------------------------------------
                    # Per-student
                    # ------------------------------------------------

                    row = {

                        "method":
                            "M1_proto_metric",

                        "outer_fold":
                            fold_id,

                        "student_group":
                            group,

                        "k_shot":
                            k,

                        "support_protocol":
                            protocol,

                        "support_size":
                            len(support_df),

                        "query_size":
                            len(query_df),

                        "accuracy":
                            metrics[
                                "accuracy"
                            ],

                        "balanced_accuracy":
                            metrics[
                                "balanced_accuracy"
                            ],

                        "macro_f1":
                            metrics[
                                "macro_f1"
                            ],

                        "majority_accuracy":
                            majority_metrics[
                                "accuracy"
                            ],

                        "majority_balanced_accuracy":
                            majority_metrics[
                                "balanced_accuracy"
                            ],

                        "majority_macro_f1":
                            majority_metrics[
                                "macro_f1"
                            ],

                        "prototype_classes_available":
                            prototype_classes_available,

                        "support_HD":
                            int(
                                support_counts[0]
                            ),

                        "support_D":
                            int(
                                support_counts[1]
                            ),

                        "support_E":
                            int(
                                support_counts[2]
                            ),

                        "support_HE":
                            int(
                                support_counts[3]
                            ),
                    }

                    per_student_rows.append(
                        row
                    )

                    # ------------------------------------------------
                    # Per class
                    # ------------------------------------------------

                    for class_id, label in enumerate(
                        LABELS
                    ):

                        per_class_rows.append(
                            {

                                "method":
                                    "M1_proto_metric",

                                "outer_fold":
                                    fold_id,

                                "student_group":
                                    group,

                                "k_shot":
                                    k,

                                "support_protocol":
                                    protocol,

                                "class":
                                    label,

                                "precision":
                                    float(
                                        metrics[
                                            "precision"
                                        ][class_id]
                                    ),

                                "recall":
                                    float(
                                        metrics[
                                            "recall"
                                        ][class_id]
                                    ),

                                "f1":
                                    float(
                                        metrics[
                                            "f1"
                                        ][class_id]
                                    ),

                                "support":
                                    int(
                                        metrics[
                                            "support"
                                        ][class_id]
                                    ),
                            }
                        )

                    # ------------------------------------------------
                    # Confusion matrix
                    # ------------------------------------------------

                    cm = metrics[
                        "confusion_matrix"
                    ]

                    for true_id in range(
                        NUM_CLASSES
                    ):

                        for pred_id in range(
                            NUM_CLASSES
                        ):

                            confusion_rows.append(
                                {

                                    "method":
                                        "M1_proto_metric",

                                    "outer_fold":
                                        fold_id,

                                    "student_group":
                                        group,

                                    "k_shot":
                                        k,

                                    "support_protocol":
                                        protocol,

                                    "true_class":
                                        LABELS[
                                            true_id
                                        ],

                                    "predicted_class":
                                        LABELS[
                                            pred_id
                                        ],

                                    "count":
                                        int(
                                            cm[
                                                true_id,
                                                pred_id
                                            ]
                                        ),
                                }
                            )

    # ========================================================
    # DATAFRAMES
    # ========================================================

    per_student_df = pd.DataFrame(
        per_student_rows
    )

    per_class_df = pd.DataFrame(
        per_class_rows
    )

    confusion_df = pd.DataFrame(
        confusion_rows
    )

    skipped_df = pd.DataFrame(
        skipped_rows
    )

    if per_student_df.empty:

        raise RuntimeError(
            "No student evaluations."
        )

    # ========================================================
    # SUMMARY
    # ========================================================

    summary_df = (
        per_student_df
        .groupby(
            [
                "support_protocol",
                "k_shot",
            ],
            as_index=False,
        )
        .agg(

            n_students=(
                "student_group",
                "nunique",
            ),

            total_query_clips=(
                "query_size",
                "sum",
            ),

            mean_accuracy=(
                "accuracy",
                "mean",
            ),

            sd_accuracy=(
                "accuracy",
                "std",
            ),

            mean_balanced_accuracy=(
                "balanced_accuracy",
                "mean",
            ),

            sd_balanced_accuracy=(
                "balanced_accuracy",
                "std",
            ),

            mean_macro_f1=(
                "macro_f1",
                "mean",
            ),

            sd_macro_f1=(
                "macro_f1",
                "std",
            ),

            mean_majority_accuracy=(
                "majority_accuracy",
                "mean",
            ),

            mean_majority_balanced_accuracy=(
                "majority_balanced_accuracy",
                "mean",
            ),

            mean_majority_macro_f1=(
                "majority_macro_f1",
                "mean",
            ),

            mean_prototype_classes_available=(
                "prototype_classes_available",
                "mean",
            ),
        )
        .sort_values(
            [
                "support_protocol",
                "k_shot",
            ]
        )
        .reset_index(
            drop=True
        )
    )

    # ========================================================
    # PER-CLASS SUMMARY
    # ========================================================

    class_summary_df = (
        per_class_df
        .groupby(
            [
                "support_protocol",
                "k_shot",
                "class",
            ],
            as_index=False,
        )
        .agg(

            mean_precision=(
                "precision",
                "mean",
            ),

            sd_precision=(
                "precision",
                "std",
            ),

            mean_recall=(
                "recall",
                "mean",
            ),

            sd_recall=(
                "recall",
                "std",
            ),

            mean_f1=(
                "f1",
                "mean",
            ),

            sd_f1=(
                "f1",
                "std",
            ),

            total_support=(
                "support",
                "sum",
            ),
        )
    )

    # ========================================================
    # HD-ONLY SUMMARY
    # ========================================================

    hd_summary_df = (
        class_summary_df[
            class_summary_df["class"]
            == "Highly Disengage"
        ]
        .copy()
        .reset_index(
            drop=True
        )
    )

    # ========================================================
    # OUTPUT
    # ========================================================

    os.makedirs(
        output_dir,
        exist_ok=True,
    )

    per_student_path = os.path.join(
        output_dir,
        "m1_per_student.csv",
    )

    per_class_path = os.path.join(
        output_dir,
        "m1_per_class.csv",
    )

    class_summary_path = os.path.join(
        output_dir,
        "m1_class_summary.csv",
    )

    hd_summary_path = os.path.join(
        output_dir,
        "m1_highly_disengage.csv",
    )

    confusion_path = os.path.join(
        output_dir,
        "m1_confusion_matrix.csv",
    )

    skipped_path = os.path.join(
        output_dir,
        "m1_skipped.csv",
    )

    summary_path = os.path.join(
        output_dir,
        "m1_summary.csv",
    )

    per_student_df.to_csv(
        per_student_path,
        index=False,
    )

    per_class_df.to_csv(
        per_class_path,
        index=False,
    )

    class_summary_df.to_csv(
        class_summary_path,
        index=False,
    )

    hd_summary_df.to_csv(
        hd_summary_path,
        index=False,
    )

    confusion_df.to_csv(
        confusion_path,
        index=False,
    )

    skipped_df.to_csv(
        skipped_path,
        index=False,
    )

    summary_df.to_csv(
        summary_path,
        index=False,
    )

    # ========================================================
    # PRINT RESULTS
    # ========================================================

    print()
    print("=" * 80)
    print(
        "M1: GLOBAL + PERSONALIZED "
        "PROTOTYPE METRIC RESULTS"
    )
    print("=" * 80)

    print(
        summary_df.to_string(
            index=False
        )
    )

    print()
    print("=" * 80)
    print(
        "M1 PER-CLASS RESULTS"
    )
    print("=" * 80)

    print(
        class_summary_df.to_string(
            index=False
        )
    )

    print()
    print("=" * 80)
    print(
        "M1 HIGHLY DISENGAGE RESULTS"
    )
    print("=" * 80)

    print(
        hd_summary_df.to_string(
            index=False
        )
    )

    print()
    print("=" * 80)
    print(
        "M1 SKIPPED CASES"
    )
    print("=" * 80)

    if skipped_df.empty:

        print(
            "None."
        )

    else:

        print(
            skipped_df[
                "reason"
            ].value_counts()
        )

    print()
    print("=" * 80)
    print(
        "M1 SAVED FILES"
    )
    print("=" * 80)

    print(
        summary_path
    )

    print(
        per_student_path
    )

    print(
        per_class_path
    )

    print(
        class_summary_path
    )

    print(
        hd_summary_path
    )

    print(
        confusion_path
    )

    print(
        skipped_path
    )

    return (
        per_student_df,
        summary_df,
        per_class_df,
        class_summary_df,
        hd_summary_df,
        confusion_df,
        skipped_df,
    )


# ============================================================
# MAIN
# ============================================================

def main():

    parser = argparse.ArgumentParser()

    parser.add_argument(
        "--json",
        type=str,
        default="/content/final_data_1.json",
    )

    parser.add_argument(
        "--folds",
        type=str,
        default=(
            "/content/"
            "cmose_h1_h2_fold_protocol.csv"
        ),
    )

    parser.add_argument(
        "--episodes",
        type=int,
        default=2000,
    )

    parser.add_argument(
        "--seed",
        type=int,
        default=42,
    )

    parser.add_argument(
        "--k-values",
        type=str,
        default="5,10,20",
    )

    parser.add_argument(
        "--output-dir",
        type=str,
        default="/content/m1_results",
    )

    parser.add_argument(
        "--device",
        type=str,
        default="auto",
        choices=[
            "auto",
            "cpu",
            "cuda",
        ],
    )

    args = parser.parse_args()

    if args.device == "cuda":

        if not torch.cuda.is_available():

            raise RuntimeError(
                "CUDA requested but unavailable."
            )

        device = torch.device(
            "cuda"
        )

    elif args.device == "cpu":

        device = torch.device(
            "cpu"
        )

    else:

        device = torch.device(
            "cuda"
            if torch.cuda.is_available()
            else "cpu"
        )

    print(
        f"Using device: {device}"
    )

    if torch.cuda.is_available():

        print(
            "GPU:",
            torch.cuda.get_device_name(0)
        )

    set_seed(
        args.seed
    )

    k_values = [
        int(v.strip())
        for v in args.k_values.split(",")
        if v.strip()
    ]

    print()
    print(
        "Loading clip data..."
    )

    clip_df = (
        load_clip_dataframe(
            args.json
        )
    )

    print(
        f"Loaded {len(clip_df)} "
        f"labelled clips from "
        f"{clip_df['group'].nunique()} "
        f"students."
    )

    print()
    print(
        "Overall class distribution:"
    )

    print(
        clip_df[
            "label"
        ].value_counts()
    )

    print()
    print(
        "Loading fold protocol..."
    )

    group_fold = (
        load_group_fold_map(
            args.folds
        )
    )

    print(
        f"Fold protocol contains "
        f"{len(group_fold)} students."
    )

    print()
    print("=" * 80)
    print(
        "STARTING M1"
    )
    print("=" * 80)

    run_experiment(
        clip_df=clip_df,
        group_fold=group_fold,
        k_values=k_values,
        seed=args.seed,
        episodes=args.episodes,
        device=device,
        output_dir=args.output_dir,
    )


# ============================================================
# COLAB-SAFE ENTRY POINT
# ============================================================

if __name__ == "__main__":

    # --------------------------------------------------------
    # Direct Colab configuration
    # --------------------------------------------------------

    JSON_PATH = (
        "/content/final_data_1.json"
    )

    FOLDS_PATH = (
        "/content/"
        "cmose_h1_h2_fold_protocol.csv"
    )

    OUTPUT_DIR = (
        "/content/m1_results"
    )

    K_VALUES = [
        5,
        10,
        20,
    ]

    EPISODES = 2000

    SEED = 42

    # --------------------------------------------------------
    # Device
    # --------------------------------------------------------

    if torch.cuda.is_available():

        DEVICE = torch.device(
            "cuda"
        )

        print(
            "CUDA available: True"
        )

        print(
            "GPU:",
            torch.cuda.get_device_name(0)
        )

    else:

        DEVICE = torch.device(
            "cpu"
        )

        print(
            "CUDA available: False"
        )

        print(
            "Using CPU."
        )

    # --------------------------------------------------------
    # Seed
    # --------------------------------------------------------

    set_seed(
        SEED
    )

    # --------------------------------------------------------
    # Load data
    # --------------------------------------------------------

    print()
    print(
        "Loading clip data..."
    )

    clip_df = (
        load_clip_dataframe(
            JSON_PATH
        )
    )

    print(
        f"Loaded {len(clip_df)} "
        f"labelled clips from "
        f"{clip_df['group'].nunique()} "
        f"students."
    )

    print()
    print(
        "Class distribution:"
    )

    print(
        clip_df[
            "label"
        ].value_counts()
    )

    print()
    print(
        "Loading fold protocol..."
    )

    group_fold = (
        load_group_fold_map(
            FOLDS_PATH
        )
    )

    print(
        f"Fold protocol contains "
        f"{len(group_fold)} students."
    )

    # --------------------------------------------------------
    # Run
    # --------------------------------------------------------

    run_experiment(
        clip_df=clip_df,
        group_fold=group_fold,
        k_values=K_VALUES,
        seed=SEED,
        episodes=EPISODES,
        device=DEVICE,
        output_dir=OUTPUT_DIR,
    )

CUDA available: True
GPU: Tesla T4

Loading clip data...
Loaded 11902 labelled clips from 103 students.

Class distribution:
label
Engage              8245
Disengage           2170
Highly Engage       1145
Highly Disengage     342
Name: count, dtype: int64

Loading fold protocol...
Fold protocol contains 82 students.

M1 OUTER FOLD 1
Training students: 65
Test students: 17

Training class distribution:
label
Engage              6291
Disengage           1601
Highly Engage        904
Highly Disengage     263
Name: count, dtype: int64

M1 meta-training (2000 episodes)
    Episode 100/2000 loss=1.6872 CE=0.9397 CON=4.3577 HD-D=0.0125 ORD=0.4423
    Episode 200/2000 loss=1.3433 CE=0.5864 CON=4.2803 HD-D=0.0869 ORD=0.3485
    Episode 300/2000 loss=1.2230 CE=0.7340 CON=4.3813 HD-D=-0.0000 ORD=0.3198
    Episode 400/2000 loss=1.0677 CE=0.4107 CON=3.9539 HD-D=0.0000 ORD=0.3067
    Episode 500/2000 loss=1.0403 CE=0.4843 CON=4.1816 HD-D=0.0000 ORD=0.2457
    Episode 600/2000 loss=0.9795 CE=0.0085

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics


M1 OUTER FOLD 2
Training students: 65
Test students: 17

Training class distribution:
label
Engage              6415
Disengage           1605
Highly Engage        893
Highly Disengage     259
Name: count, dtype: int64

M1 meta-training (2000 episodes)
    Episode 100/2000 loss=1.7742 CE=0.7369 CON=4.5173 HD-D=0.0000 ORD=0.5196
    Episode 200/2000 loss=1.2895 CE=0.8249 CON=4.7690 HD-D=0.0647 ORD=0.3840
    Episode 300/2000 loss=1.2991 CE=0.8992 CON=4.2689 HD-D=0.0000 ORD=0.2329
    Episode 400/2000 loss=1.1456 CE=0.1509 CON=3.9658 HD-D=0.0000 ORD=0.0327
    Episode 500/2000 loss=1.0778 CE=0.2607 CON=4.0180 HD-D=0.0000 ORD=0.1925
    Episode 600/2000 loss=0.9010 CE=0.2743 CON=3.9709 HD-D=0.0000 ORD=0.2302
    Episode 700/2000 loss=0.8926 CE=0.3407 CON=4.4682 HD-D=-0.0000 ORD=0.5095
    Episode 800/2000 loss=0.8020 CE=0.0536 CON=3.4006 HD-D=0.0000 ORD=0.1357
    Episode 900/2000 loss=0.7702 CE=0.0462 CON=4.2787 HD-D=-0.0000 ORD=0.2073
    Episode 1000/2000 loss=0.7922 CE=0.0299 CON=3.69

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics


M1 OUTER FOLD 3
Training students: 66
Test students: 16

Training class distribution:
label
Engage              6459
Disengage           1628
Highly Engage        872
Highly Disengage     251
Name: count, dtype: int64

M1 meta-training (2000 episodes)
    Episode 100/2000 loss=1.7374 CE=1.6882 CON=4.6507 HD-D=0.0000 ORD=1.0818
    Episode 200/2000 loss=1.3342 CE=0.5248 CON=4.0940 HD-D=0.0000 ORD=0.8864
    Episode 300/2000 loss=1.2518 CE=0.6107 CON=4.4064 HD-D=0.0223 ORD=0.2942
    Episode 400/2000 loss=1.1237 CE=0.2291 CON=4.1933 HD-D=0.0000 ORD=0.4496
    Episode 500/2000 loss=1.0466 CE=0.6981 CON=4.6807 HD-D=0.0000 ORD=0.3260
    Episode 600/2000 loss=0.9895 CE=0.4733 CON=4.1000 HD-D=0.0000 ORD=0.2692
    Episode 700/2000 loss=0.9164 CE=0.3282 CON=3.9919 HD-D=0.0036 ORD=0.3931
    Episode 800/2000 loss=0.7995 CE=0.1675 CON=4.2614 HD-D=0.0029 ORD=0.1030
    Episode 900/2000 loss=0.7834 CE=0.3243 CON=4.0857 HD-D=0.0774 ORD=0.2023
    Episode 1000/2000 loss=0.7540 CE=0.2876 CON=4.0320

...


M1 OUTER FOLD 4
Training students: 66
Test students: 16

Training class distribution:
label
Engage              6111
Disengage           1623
Highly Engage        815
Highly Disengage     237
Name: count, dtype: int64

M1 meta-training (2000 episodes)
    Episode 100/2000 loss=1.7144 CE=0.1890 CON=3.8346 HD-D=0.0000 ORD=0.3864
    Episode 200/2000 loss=1.3196 CE=0.3044 CON=4.4142 HD-D=0.1394 ORD=0.1317
    Episode 300/2000 loss=1.2223 CE=0.3266 CON=4.1627 HD-D=0.0000 ORD=0.6883
    Episode 400/2000 loss=1.0732 CE=0.1090 CON=4.1000 HD-D=0.0000 ORD=0.2129
    Episode 500/2000 loss=1.0011 CE=0.6957 CON=4.3794 HD-D=0.0114 ORD=0.4545
    Episode 600/2000 loss=0.9633 CE=0.2146 CON=3.6897 HD-D=0.0000 ORD=0.5773
    Episode 700/2000 loss=0.9713 CE=0.9055 CON=3.8013 HD-D=0.0439 ORD=0.3853
    Episode 800/2000 loss=0.8515 CE=0.3907 CON=3.9499 HD-D=0.0750 ORD=0.2266
    Episode 900/2000 loss=0.8298 CE=0.0454 CON=4.1470 HD-D=0.0000 ORD=0.1031
    Episode 1000/2000 loss=0.7424 CE=0.1535 CON=4.0136

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics


M1 OUTER FOLD 5
Training students: 66
Test students: 16

Training class distribution:
label
Engage              6568
Disengage           1727
Highly Engage        912
Highly Disengage     270
Name: count, dtype: int64

M1 meta-training (2000 episodes)
    Episode 100/2000 loss=1.6767 CE=0.4366 CON=4.5475 HD-D=0.0000 ORD=0.1768
    Episode 200/2000 loss=1.4058 CE=0.4770 CON=4.1712 HD-D=0.0000 ORD=0.7329
    Episode 300/2000 loss=1.2312 CE=0.7466 CON=4.1128 HD-D=0.0740 ORD=0.5250
    Episode 400/2000 loss=1.1402 CE=0.7427 CON=4.4759 HD-D=0.0000 ORD=0.8575
    Episode 500/2000 loss=0.9890 CE=0.1802 CON=4.4455 HD-D=-0.0000 ORD=0.5048
    Episode 600/2000 loss=0.9800 CE=0.6340 CON=4.2126 HD-D=0.0104 ORD=0.4363
    Episode 700/2000 loss=0.9317 CE=0.5917 CON=4.2821 HD-D=0.0000 ORD=0.4387
    Episode 800/2000 loss=0.8619 CE=0.4531 CON=4.3559 HD-D=0.0229 ORD=0.2435
    Episode 900/2000 loss=0.7980 CE=0.3072 CON=4.2186 HD-D=0.0000 ORD=0.3850
    Episode 1000/2000 loss=0.7760 CE=0.2061 CON=3.840

/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")
/usr/local/lib/python3.13/dist-packages/sklearn/metrics


M1: GLOBAL + PERSONALIZED PROTOTYPE METRIC RESULTS
support_protocol  k_shot  n_students  total_query_clips  mean_accuracy  sd_accuracy  mean_balanced_accuracy  sd_balanced_accuracy  mean_macro_f1  sd_macro_f1  mean_majority_accuracy  mean_majority_balanced_accuracy  mean_majority_macro_f1  mean_prototype_classes_available
           early       5          82              11016       0.639220     0.192193                0.422473              0.119627       0.312337     0.089222                0.708270                         0.307927                0.205566                          1.902439
           early      10          82              10606       0.635142     0.178951                0.436256              0.125889       0.321302     0.091154                0.710367                         0.315041                0.205928                          2.231707
           early      20          82               9786       0.620855     0.170342                0.469567              0.151654